In [ ]:
import os
import torch
import pytorch_lightning as pl
# need transformers version 4.53.1
from transformers import get_scheduler, AutoModelForCausalLM, AutoProcessor, AutoConfig  
from florence_2_base_ft import processing_florence2
from peft import LoraConfig, get_peft_model, PeftModel, PeftConfig
from pytorch_lightning import Trainer
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import MultiLabelBinarizer
from checkpoint_callback import CustomModelCheckpoint
from sklearn.model_selection import train_test_split
from torchvision.transforms.functional import to_pil_image
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import albumentations as A
import ast
import torchvision.transforms as T
import supervision as sv
import cv2
from PIL import Image
import yaml
import pydicom
import io
import itertools
from datetime import datetime
import tensorflow as tf
from tensorflow import keras
from pydicom.pixel_data_handlers.util import apply_voi_lut
from difflib import get_close_matches

# os.environ["TOKENIZERS_PARALLELISM"] = "false"

# %env PYTORCH_NO_CUDA_MEMORY_CACHING=1

In [ ]:
torch.cuda.empty_cache()

In [ ]:
# # allows for tensorboard to be opened

# %reload_ext tensorboard
# %tensorboard --logdir=lightning_logs/

# # cd C:\Users\Mark\Documents\GitHub\diss-cxr-xception\florence_attempt
# # python -m tensorboard.main --logdir=lightning_logs/ 

In [ ]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [ ]:
def load_config(config_path):
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    return config

In [ ]:
CLASSES = ['No finding', 'Pleural thickening', 'Aortic enlargement', 'Pulmonary fibrosis', 'Cardiomegaly', 'Nodule or Mass', 'Lung Opacity', 'Other lesion', 'Pleural effusion', 'ILD', 'Infiltration', 'Calcification', 'Consolidation', 'Atelectasis', 'Rib fracture', 'Mediastinal shift', 'Enlarged PA', 'Pneumothorax', 'Emphysema', 'Lung cavity', 'Lung cyst', 'Clavicle fracture', 'Edema']
CLASSES = [cls.lower() for cls in CLASSES]
CLASSES

In [ ]:
# loads the cofig for running the model
config_path = "configs/experiment.yaml"
config = load_config(config_path)

In [ ]:
# uses GPU if able
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# DEVICE = torch.device("cpu")
# DEVICE = torch.device("xpu")

# this revision fixes the GenerationMixin import issue 
REVISION = 'refs/pr/24'  # florence-2-base-ft
# REVISION = 'refs/pr/38'  # florence-2-large-ft

MODEL_NAME = "microsoft/Florence-2-base-ft"

### initialising the model
# downloads the model config from hugging face 
config_model = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
config_model.vision_config.model_type = "davit"
# model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code = True, config = config_model,revision = REVISION).to(DEVICE)
# builds generic model using florence2
# model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code = True, config = config_model, revision = REVISION).to(DEVICE)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code = True, revision = REVISION).to(DEVICE)

# defines the processor to be used (florence2)

# processor = processing_florence2.Florence2Processor.from_pretrained("./florence_2_base_ft")
# processor.image_processor.size = config['model']['processor']['image_size']
# processor.image_processor.crop_size = config['model']['processor']['crop_size']
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True, revision = REVISION)

In [ ]:
# # freeze more layers 
# for name, param in model.named_parameters():
#     if 'vision_tower' in name or 'language_model' in name:
#         param.requires_grad = False
        
# # only unfreeze task-specific layers
# for name, param in model.named_parameters():
#     if 'grounding_head' in name:
#         param.requires_grad = True

In [ ]:
torch.cuda.empty_cache()

In [ ]:
# annotations = pd.read_csv("E:/vinbigdata_xrays/vinbigdata/train_original.csv")  # full dataset
# annotations = pd.read_csv("E:/vinbigdata_xrays/vinbigdata/train_quarter.csv")  # quarter of the dataset
annotations = pd.read_csv("E:/vinbigdata_xrays/vinbigdata/train_10_percent.csv")  # 10 percent of dataset
# annotations = pd.read_csv("E:/vinbigdata_xrays/vinbigdata/train_1_percent.csv")  # 1 percent of dataset
# annotations = pd.read_csv("E:/vinbigdata_xrays/vinbigdata/train_truncated.csv")  # only 40 items, used for testing if model works
# annotations = pd.read_csv("E:/vinbigdata_xrays/vinbigdata/train_all_same_pred.csv")  # all labels are Cardiomegaly, only 40 items
# annotations = pd.read_csv("E:/vinbigdata_xrays/vinbigdata/train_all_same_rows.csv")  # all labels are the same, 50 items

# drop all rows that contain 'no finding'
# no finding rows don't have bboxes
annotations = annotations.drop(annotations[annotations['class_name'] == 'No finding'].index)
annotations['class_name'] = annotations['class_name'].str.lower().replace('nodule/mass', 'nodule or mass')

print(annotations['class_name'].value_counts())

CLASSES = annotations['class_name'].unique().tolist()
# df.word.value_counts()['myword']

# create data splits
train_data, rest_data = train_test_split(annotations, train_size=0.8, shuffle=False)
validation_data, test_data = train_test_split(rest_data, test_size=0.5, shuffle=False)

# add split column
train_data['split'] = 'train'
validation_data['split'] = 'validate'
test_data['split'] = 'test'

# combine and save
split_annotations = pd.concat([train_data, validation_data, test_data]).reset_index(drop=True)
split_annotations.to_csv('E:/vinbigdata_xrays/vinbigdata/train.csv', index=False)

In [ ]:
def evaluate_results(model, inputs, processor, answers, images, batch_idx, questions, original_sizes):
    bounding_box_annotator = sv.BoxAnnotator(color_lookup=sv.ColorLookup.INDEX)
    label_annotator = sv.LabelAnnotator(color_lookup=sv.ColorLookup.INDEX)
    color_annotator = sv.ColorAnnotator(color_lookup=sv.ColorLookup.INDEX)
    
    # print(f"Input IDs shape: {inputs['input_ids'].shape}")
    # print(f"Input Pixel Values shape: {inputs['pixel_values'].shape}")

    # print(f"Tokenized input text: {processor.tokenizer.batch_decode(inputs['input_ids'], skip_special_tokens=True)}")


    generated_ids = model.generate(input_ids=inputs["input_ids"], 
                                   pixel_values=inputs["pixel_values"], 
                                   max_new_tokens=1024,  # set to highest possible
                                   do_sample=False,
                                   num_beams=1,
                                   output_scores=True,
                                   return_dict_in_generate=True
    )
    # print(f"model.generate outputs: {generated_ids}")
    # print("input text:", questions[0])
    # print("input pixel value shapes:", inputs["pixel_values"].shape)

    # model.generate(input_ids=inputs["input_ids"],pixel_values=inputs["pixel_values"], max_new_tokens=300,  num_beams=3)
    
    # decode the predicted answers
    generated_text = processor.batch_decode(generated_ids.sequences, skip_special_tokens=False, clean_up_tokenization_spaces=True)

    print(f"Generated text: {generated_text}")

    targets = []
    predictions = []
    img_list = []
    pred_text_list = []
    gt_text_list = []
    pred_label = []
    gt_label = []
    captions = []
    
    for i, text in enumerate(generated_text):
        # # gets the original dicom resolution
        # img_id = questions[i].split()[0]  # or however you can extract image_id
        # dicom_path = os.path.join("path/to/original/dicoms", f"{img_id}.dicom")
        # ds = pydicom.dcmread(dicom_path)
        # original_height, original_width = ds.pixel_array.shape
        # original_resolution = (original_width, original_height)
        
        # print(f"images: {images}")

        # original_resolution = original_sizes[i]

        # # answer = processor.post_process_generation(text, task='<CAPTION_TO_PHRASE_GROUNDING>', image_size=images[i].shape[1:3])  # image shape is C, W, H
        # # gt_answer = processor.post_process_generation(answers[i], task='<CAPTION_TO_PHRASE_GROUNDING>', image_size=images[i].shape[1:3])
        # answer = processor.post_process_generation(text, task='<CAPTION_TO_PHRASE_GROUNDING>', image_size=original_resolution)  # image shape is C, W, H
        # gt_answer = processor.post_process_generation(answers[i], task='<CAPTION_TO_PHRASE_GROUNDING>', image_size=original_resolution)
        # # answer = processor.post_process_generation(text, task='<OPEN_VOCABULARY_DETECTION>', image_size=images[i].shape[:2])
        # # gt_answer = processor.post_process_generation(answers[i], task='<OPEN_VOCABULARY_DETECTION>', image_size=images[i].shape[:2])

        resized_resolution = images[i].shape[1:3]  # (W, H)

        answer = processor.post_process_generation(text, task='<CAPTION_TO_PHRASE_GROUNDING>', image_size=resized_resolution)
        gt_answer = processor.post_process_generation(answers[i], task='<CAPTION_TO_PHRASE_GROUNDING>', image_size=resized_resolution)

        # TODO: check if original_resolution here is correct in terms of yesterdays resolution checking
        
        print(f"Answer: {answer}")
        print(f"gt_answer: {gt_answer}")

        # Create detections for both the ground truth and predicted answers
        gt = sv.Detections.from_lmm(sv.LMM.FLORENCE_2, gt_answer, resolution_wh=images[i].shape[1:3])

        # gets the ID of the class name
        gt.class_id = np.array([CLASSES.index(class_name) for class_name in gt['class_name']])
        
        # gets the label of the class in terms of its name
        class_label = gt['class_name'][0]

        # print(f"Class label: {class_label}")

        # converts the image from a Tensor to a PIL image and then to a numpy array
        pil_image = to_pil_image(images[i].clone())
        image_with_ground_truth = bounding_box_annotator.annotate(pil_image, gt)
        image_with_ground_truth = label_annotator.annotate(image_with_ground_truth, gt)
        # display(image_with_ground_truth)

        pred_text_list.append(answer)
        gt_text_list.append(gt_answer)
        
        # prediction = sv.Detections.from_lmm(sv.LMM.FLORENCE_2, answer, resolution_wh=images[i].shape[1:3])
        prediction = sv.Detections.from_lmm(sv.LMM.FLORENCE_2, answer, resolution_wh=resized_resolution)
        class_names = prediction['class_name']
        xyxys = prediction.xyxy

        # print(f"prediction: {prediction}")
        # print(f"class names: {class_names}")
        # print(f"xyxys: {xyxys}")

        if class_names is not None and len(class_names) > 0:
            prediction = prediction[~np.char.startswith(class_names, 'mark')]
            class_names = prediction['class_name']
            xyxys = prediction.xyxy

            corrected_class_names = []

            # uses get_close_matches to get predictions that are close enough to a class name
            for name in class_names:
                match = get_close_matches(name, CLASSES, n=1, cutoff=0.5)
                corrected_class_names.append(match[0] if match else name)

            class_names = corrected_class_names

            # some error handling when there aren't enough bounding boxes for predictions
            # typically occured when the model would predict multiple objects
            if len(xyxys) != len(class_names):
                if len(xyxys) == 1 and len(class_names) > 1:
                    xyxys = np.repeat(xyxys, len(class_names), axis=0)
                elif len(xyxys) < len(class_names):
                    print(f"Not enough boxes for predicted classes at index {i}. Skipping.")
                    continue
                elif len(xyxys) > len(class_names):
                    class_names += ["no finding"] * (len(xyxys) - len(class_names))

            # builds the prediction using supervision's Detections
            try:
                prediction = sv.Detections(
                    xyxy=xyxys,
                    class_id=np.array([CLASSES.index(cn) if cn in CLASSES else CLASSES.index("no finding") for cn in class_names]),
                    confidence=np.ones(len(xyxys))
                )
            except Exception as e:
                print(f"Failed to build prediction at index {i}: {e}")
                continue

            pred_label.append(class_names)
            gt_label.append(gt['class_name'].tolist())

            targets.append(gt)
            predictions.append(prediction)

            if i < 10:
                image_with_predictions = bounding_box_annotator.annotate(image_with_ground_truth.copy(), prediction)
                image_with_predictions = color_annotator.annotate(image_with_predictions, prediction)
                img_list.append(image_with_predictions)
                captions.append(f"Question: {questions[i]}\nGround truth: {gt_answer}\nPredicted: {answer}")

                display(image_with_predictions)

        else:
            targets.append(gt)
            predictions.append(sv.Detections.empty())
            pred_label.append([])
            gt_label.append(gt['class_name'])
            if i < 10:
                img_list.append(image_with_ground_truth)
                captions.append(f"Question: {questions[i]}\nGround truth: {gt_answer}\nPredicted: {answer}")

    return {
        "res_samples": img_list,
        "predictions": predictions,
        "targets": targets,
        "text_pred_answer": pred_text_list,
        "text_gt_answer": gt_text_list,
        "pred_label": pred_label,
        "gt_label": gt_label,
        "captions": captions
    }
    # mean_average_precision = sv.MeanAveragePrecision.from_detections(predictions=predictions,targets=targets)



        # print(confusion_matrix.matrix)

    #     # Ensure valid class names
    #     prediction['class_name'] = correct_class_names(prediction['class_name'], CLASSES)
    #     prediction = prediction[np.isin(prediction['class_name'], CLASSES)]

    #     # Assign class_id and confidence for the prediction
    #     prediction.class_id = np.array([CLASSES.index(class_name) for class_name in prediction['class_name']])
    #     prediction.confidence = np.ones(len(prediction))

    #     # Annotate images with ground truth and predictions
    #     image_with_predictions = annotate_image(images[i], prediction, bounding_box_annotator, label_annotator)
    #     image_with_ground_truth = annotate_image(images[i], gt, bounding_box_annotator, label_annotator)

    #     # Convert to PIL images for saving
    #     image_with_ground_truth = Image.fromarray(image_with_ground_truth.astype(np.uint8))
    #     image_with_predictions = Image.fromarray(image_with_predictions.astype(np.uint8))

    #     # Combine and save images
    #     combined_image = combine_images(image_with_ground_truth, image_with_predictions)
    #     combined_image.save(f"./combined_image_{batch_idx}_{i}.png")

    #     processed_predictions.append((image_with_ground_truth, image_with_predictions))  # You could also store metrics here

    # return processed_predictions
    # return 'ok'



In [ ]:
# converts DICOM files to np arrays
# copied from https://www.kaggle.com/code/raddar/convert-dicom-to-np-array-the-correct-way
def read_xray(path, voi_lut = True, fix_monochrome = True, target_size=(128, 128)):
    # try reading image as a DICOM file
    try:
        dicom = pydicom.dcmread(path)

        # VOI LUT (if available by DICOM device) is used to transform raw DICOM data to "human-friendly" view
        if voi_lut:
            data = apply_voi_lut(dicom.pixel_array, dicom)
        else:
            data = dicom.pixel_array
                
        # depending on this value, X-ray may look inverted - fix that:
        if fix_monochrome and dicom.PhotometricInterpretation == "MONOCHROME1":
            data = np.amax(data) - data
            
    # file isn't a DICOM file, most likely png/jpg/etc
    except:
        data = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if data is None:
            raise ValueError(f"File at {path} is neither a valid DICOM nor an image.")


    # normalize to [0, 255]
    data = data - np.min(data)
    data = data / np.max(data)
    data = (data * 255).astype(np.uint8)

    # add padding to make the image square
    h, w = data.shape
    if h != w:
        max_dim = max(h, w)
        padded_image = np.zeros((max_dim, max_dim), dtype=np.uint8)
        padded_image[(max_dim - h) // 2:(max_dim - h) // 2 + h,
                     (max_dim - w) // 2:(max_dim - w) // 2 + w] = data
        data = padded_image

    # resize to target size
    data = cv2.resize(data, target_size, interpolation=cv2.INTER_LINEAR)

    # add channel dimension (C=1)
    data = np.expand_dims(data, axis=0)

    return data

In [ ]:
## checking DICOM file dimensions

# input_dir = "E:/vinbigdata_xrays/vinbigdata/train"

# for filename in os.listdir(input_dir):
#     input_path = os.path.join(input_dir, filename)
#     dicom = pydicom.dcmread(input_path)
#     rows = dicom.Rows
#     cols = dicom.Columns
#     print(f"{filename}: {cols} x {rows}")

In [ ]:
# converts DICOM files to PNG and saves them
def process_and_save_xrays(input_dir, output_dir, voi_lut=True, fix_monochrome=True, target_size=(128, 128)):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    for filename in os.listdir(input_dir):
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, f"{os.path.splitext(filename)[0]}.png")
        
        try:
            # Read the DICOM or image file
            dicom = pydicom.dcmread(input_path)

            # Apply VOI LUT if applicable
            if voi_lut:
                data = apply_voi_lut(dicom.pixel_array, dicom)
            else:
                data = dicom.pixel_array

            # Fix monochrome if necessary
            if fix_monochrome and dicom.PhotometricInterpretation == "MONOCHROME1":
                data = np.amax(data) - data
        except Exception as e:
            print(e)
            # Fall back to reading as an image
            data = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)
            if data is None:
                print(f"Skipping invalid file: {input_path}")
                continue
        
        # Normalize to [0, 255]
        data = data - np.min(data)
        data = data / np.max(data)
        data = (data * 255).astype(np.uint8)

        # Add padding to make the image square
        h, w = data.shape
        if h != w:
            max_dim = max(h, w)
            padded_image = np.zeros((max_dim, max_dim), dtype=np.uint8)
            padded_image[(max_dim - h) // 2:(max_dim - h) // 2 + h,
                         (max_dim - w) // 2:(max_dim - w) // 2 + w] = data
            data = padded_image

        # resize to target size
        data = cv2.resize(data, target_size, interpolation=cv2.INTER_LINEAR)

        # save as PNG
        cv2.imwrite(output_path, data)
        print(f"saved image to {output_path}")

In [ ]:
# import os
# import cv2
# import pydicom
# import numpy as np
# from pydicom.pixel_data_handlers.util import apply_voi_lut

# input_dir = "C:/Users/markf/Downloads/CXR/"
# output_dir = "C:/Users/markf/Downloads/CXR/PNG/"

# input_dir = "E:/vinbigdata_xrays/vinbigdata/train/dicom/"
# output_dir = "E:/vinbigdata_xrays/vinbigdata/train/png/"

# process_and_save_xrays(input_dir, output_dir, target_size=(512, 512))


In [ ]:
class VindrDataset(Dataset):
    def __init__(self, img_root, dicom_root, annotation_csv, split='train', data_pct=1.0, transform=None):
        self.img_root = img_root
        self.dicom_root = dicom_root
        self.transform = transform
        self.annotations = pd.read_csv(annotation_csv)
        
        # Check if split is valid
        if split not in ['train', 'test', 'validate']:
            raise ValueError(f"Invalid split: {split}. Expected one of ['train', 'test', 'validate'].")
        
        # Check if data_pct is valid
        if not (0 < data_pct <= 1):
            raise ValueError(f"data_pct should be in the range (0, 1], got {data_pct}")
        
        # filtering and splitting
        self.annotations = self.annotations[self.annotations['split'] == split].reset_index(drop=True)
        
        # if split == 'train':
        #     self.annotations = self.annotations[(self.annotations['split'] == 'train') | (self.annotations['split'] == 'validate')].reset_index(drop=True)
        # else:
        #     self.annotations = self.annotations[self.annotations['split'] == split].reset_index(drop=True)

        if self.annotations.empty:
            raise ValueError(f"No data available for split: {split}")

        # sample data based on data_pct
        if data_pct < 1.0:
            sampled_indices = np.random.choice(len(self.annotations), size=int(len(self.annotations) * data_pct), replace=False)
            self.annotations = self.annotations.iloc[sampled_indices].reset_index(drop=True)

        # # grouping annotations by image_id
        # # each image has multiple annotations by different radiologistis for different anatomical defects
        # self.grouped_annotations = self.annotations.groupby('image_id')
        # self.image_ids = list(self.grouped_annotations.groups.keys())

        print(f"Loaded {len(self.annotations)} samples for split: {split}")

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        row = self.annotations.iloc[idx]  # gets the row for this item in the dataset

        # loads image
        img_id = self.annotations.iloc[idx]['image_id']

        # img_path = os.path.join(self.img_root, f"{img_id}.dicom")
        # # image = np.array(Image.open(img_path).convert("RGB"))
        # image = read_xray(img_path)

        # loads image (128x128 png)
        png_path = os.path.join(self.img_root, f"{img_id}.png")
        image = Image.open(png_path).convert("RGB")
        # resized_width, resized_height = image.size  # should be 128x128
        # image = np.array(image)
        # image = read_xray(img_path)

        # print(f"Resized: {resized_height, resized_width}")

        # TODO: check if images are actually getting their size processed correctly

        # loads original dicom to get original dimensions
        dicom_path = os.path.join(self.dicom_root, f"{img_id}.dicom")
        ds = pydicom.dcmread(dicom_path)
        original_height, original_width = ds.pixel_array.shape

        # 2000 x 2336 or something

        # when the dicom images are converted to png, they have padding added to make them square
        # need to account for this, otherwise the annotation csv's bounding boxes don't line up
        max_dim = max(original_height, original_width)  # gets the largest dimension (width or height)
        pad_top = (max_dim - original_height) // 2  # gets the amount of padding at the top
        pad_left = (max_dim - original_width) // 2  # same for the sides

        resized_width, resized_height = image.size

        # compute scale factors based on png and dicom dimensions
        # needed to ensure sizes are normalised
        x_scale = resized_width / max_dim
        y_scale = resized_height / max_dim

        # shifts bounding box by the padding values and then scale 
        x_min = (row['x_min'] + pad_left) * x_scale
        x_max = (row['x_max'] + pad_left) * x_scale
        y_min = (row['y_min'] + pad_top) * y_scale
        y_max = (row['y_max'] + pad_top) * y_scale
        boxes = [[x_min, y_min, x_max, y_max]]

        # print(f"Boxes {boxes}")

        # # boxes = ast.literal_eval(self.annotations.iloc[idx]['bboxes'])
        # boxes = [[self.annotations.iloc[idx]['x_min'], 
        #          self.annotations.iloc[idx]['y_min'],
        #          self.annotations.iloc[idx]['x_max'],
        #          self.annotations.iloc[idx]['y_max']]]
        
        # # get all annotations for this image
        # image_annotations = self.grouped_annotations.get_group(img_id)

        # # extract bounding boxes and class names for this image
        # boxes = []
        # class_names = []
        # for _, row in image_annotations.iterrows():
        #     boxes.append([row['x_min'], row['y_min'], row['x_max'], row['y_max']])
        #     class_name = row['class_name'].lower().replace('vindrcxr/', '')
        #     if class_name == 'Nodule/Mass':
        #         class_name = 'Nodule or Mass'
        #     class_names.append(class_name)
            
        # print(f"Image {img_id} has {len(boxes)} bounding boxes.")

        class_names = self.annotations.iloc[idx]['class_name'].lower().replace('vindrcxr/', '')
        
        if class_names == 'Nodule/Mass':
            class_names = 'Nodule or Mass'

        # converts the PIL image to a numpy array
        image = np.array(image)

        # checks if image is greyscale and converts to 3 channels
        if image.shape[0] == 1: 
            image = np.repeat(image, 3, axis=0)

        # makes the image a torch tensor
        if not isinstance(image, torch.Tensor):
            # permutes to C, H, W format too
            image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1) / 255.0   # TODO: normalising here too? prolly shouldn't

        return {
            'image': image,
            'boxes': boxes, 
            'label': class_names,
            'original_size': (original_width, original_height)  # returns the original size of the (dicom) image
        }
    

In [ ]:

class DetInstructDataset_vindr(Dataset):
    # def __init__(self, base_dataset, scale_factor=1000, task="<OPEN_VOCABULARY_DETECTION>", task_prompt="{input}", max_classes=5, max_invalid_cls=2, use_definition=True):
    # def __init__(self, base_dataset, task="<CAPTION_TO_PHRASE_GROUNDING>", task_prompt="Locate the phrases in the caption: {input}.", use_definition=True):
    def __init__(self, base_dataset, task="<CAPTION_TO_PHRASE_GROUNDING>", task_prompt="{input}", use_definition=True):
        self.base_dataset = base_dataset
        self.task_prompt = task_prompt
        self.task = task
        self.scale_factor = 1000 
        self.definition = yaml.safe_load(open('configs/vindr_definition.yaml')) 
        self.use_definition = use_definition
        print('Using definition:', self.use_definition)


    def __len__(self):
        return len(self.base_dataset)

    # def normalize_coordinates(self, bbox, image_shape):
    #     x1, y1, x2, y2 = bbox
    #     h, w = image_shape[:2] # image shape (H, W, C)
    #     normalized_x1 = int((x1 / w) * self.scale_factor)
    #     normalized_y1 = int((y1 / h) * self.scale_factor)
    #     normalized_x2 = int((x2 / w) * self.scale_factor)
    #     normalized_y2 = int((y2 / h) * self.scale_factor)
    #     return f"<loc_{normalized_x1}><loc_{normalized_y1}><loc_{normalized_x2}><loc_{normalized_y2}>"

    def normalize_coordinates(self, bbox, image_shape):
        if not isinstance(bbox, (list, tuple)) or len(bbox) != 4:
            raise ValueError(f"Invalid bounding box: {bbox}. Expected a list or tuple of four values.")
        x1, y1, x2, y2 = bbox
        h, w = image_shape[1:3]  # image shape (C, H, W)
        normalized_x1 = int((x1 / w) * self.scale_factor)
        normalized_y1 = int((y1 / h) * self.scale_factor)
        normalized_x2 = int((x2 / w) * self.scale_factor)
        normalized_y2 = int((y2 / h) * self.scale_factor)
        return f"<loc_{normalized_x1}><loc_{normalized_y1}><loc_{normalized_x2}><loc_{normalized_y2}>"


    def __getitem__(self, idx):
        # Get data from the base dataset
        sample = self.base_dataset[idx]
        image = sample['image']
        bounding_boxes = sample['boxes']
        det_obj = sample['label']
        original_size = sample['original_size']
        
        definition = self.definition[det_obj]
        # definition = [self.definition[class_name] for class_name in det_obj if class_name in self.definition]

        answer = []
        for bbox in bounding_boxes:
            if len(bbox) != 4:
                raise ValueError(f"Bounding box {bbox} has invalid length: {len(bbox)}") # each xray has a set of bounding boxes
            locs = self.normalize_coordinates(bbox, image.shape)
            answer.append(f"{det_obj}{locs}")
        final_answer = "".join(answer)
        # print(f"answer: {final_answer}")

        if self.use_definition:
            # det_obj = '{}, which means {}'.format(det_obj, definition)
            det_obj = 'A chest X-ray showing {}, which means {}'.format(det_obj, definition)  # TODO: maybe try to use this prompt
            # det_obj = definition
        
        # generate the task-specific prompt
        task_prompt = self.task_prompt.format(input=det_obj)
        task_prompt = self.task + task_prompt
        # print(f"Task prompt: {task_prompt}")
        

        return {    
            'image': image,
            'question': task_prompt,
            'answer': final_answer,
            'task': self.task,
            'original_size': original_size
        }



In [ ]:
class VinderDataLoaderManager(pl.LightningDataModule):
    def __init__(self, config):
        super().__init__()
        # Extract parameters from config
        self.img_root = config.get("img_root")
        self.dicom_root = config.get("dicom_root")
        self.annotation_csv = config.get("annotation_csv")
        self.batch_size = config.get("batch_size", 8)
        self.data_pct = config.get("data_pct", 1.0)
        self.num_workers = config.get("num_workers", 0)
        self.device = config.get("device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
        self.use_definition = config["use_definition"]
        
        # Use the passed processor or initialize a default one


    def collate_fn(self, batch):
        # # Unzip the batch into questions, answers, and images
        # questions = [item['question'] for item in batch]
        # answers = [item['answer'] for item in batch]
        # images = [item['image'] for item in batch]
        # tasks = [item['task'] for item in batch]
    
        # return images, questions, answers, tasks

        images = torch.stack([item['image'] for item in batch])  # stack into a single tensor
        questions = [item['question'] for item in batch]
        answers = [item['answer'] for item in batch]
        tasks = [item['task'] for item in batch]
        original_sizes = [item['original_size'] for item in batch]
        return images, questions, answers, tasks, original_sizes
            
    
    def create_dataloader(self, split):
        # Initialize the base dataset
        base_dataset = VindrDataset(
            img_root=self.img_root,
            dicom_root=self.dicom_root,
            annotation_csv=self.annotation_csv,
            split=split,
            data_pct=self.data_pct,
            transform=None
        )

        # Initialize the Multi_task_Instructer dataset
        multi_task_dataset = DetInstructDataset_vindr(
            base_dataset=base_dataset,
            use_definition=self.use_definition
        )

        # Create DataLoader
        return DataLoader(
            multi_task_dataset,
            batch_size=self.batch_size,
            collate_fn=self.collate_fn, 
            num_workers=self.num_workers,
            shuffle=True if split == 'train' else False
            # persistent_workers=True  # use if using num_workers>0
        )
    
    def train_dataloader(self):
        return self.create_dataloader(split='train')

    def val_dataloader(self):
        return self.create_dataloader(split='validate')

    def test_dataloader(self):
        return self.create_dataloader(split='test')

# class VinderDataLoaderManager(pl.LightningDataModule):
#     def __init__(self, config):
#         super().__init__()
#         # self.config = config

#         # Extract parameters from config
#         self.img_root = config.get("img_root")
#         self.annotation_csv = config.get("annotation_csv")
#         self.batch_size = config.get("batch_size", 8)
#         self.data_pct = config.get("data_pct", 1.0)
#         self.num_workers = config.get("num_workers", 0)
#         self.device = config.get("device", torch.device("cuda" if torch.cuda.is_available() else "cpu"))
#         # self.device = config.get("device", "cpu")
#         # self.device = config.get("device", "xpu")
#         self.use_definition = config["use_definition"]

#         self.train_dataset = VindrDataset(self.img_root, self.annotation_csv, split='train', data_pct=self.data_pct)
#         self.val_dataset = VindrDataset(self.img_root, self.annotation_csv, split='validate', data_pct=self.data_pct)
#         self.test_dataset = VindrDataset(self.img_root, self.annotation_csv, split='test', data_pct=self.data_pct)

#     # def setup(self, stage=None):
        

#     def collate_fn(self, batch):
#         images = torch.stack([item['image'] for item in batch])
#         questions = [item['question'] for item in batch] if 'question' in batch[0] else None
#         answers = [item['answer'] for item in batch] if 'answer' in batch[0] else None
#         tasks = [item['task'] for item in batch] if 'task' in batch[0] else None
#         return images, questions, answers, tasks

#     def train_dataloader(self):
#         return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers, pin_memory=True, collate_fn=self.collate_fn)

#     def val_dataloader(self):
#         return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers, pin_memory=True, collate_fn=self.collate_fn)

#     def test_dataloader(self):
#         return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers, pin_memory=True, collate_fn=self.collate_fn)


In [ ]:
# class FlorenceLightningModel(pl.LightningModule):
#     def __init__(self, model, processor, lr=1e-6, num_training_steps=None):
#         super(FlorenceLightningModel, self).__init__()
#         self.model = model
#         self.processor = processor
#         self.lr = float(lr)
#         self.num_training_steps = num_training_steps
#         self.test_outputs = []
#         self.valid_outputs = []

#     def training_step(self, batch, batch_idx):
#         images, questions, answers, tasks = batch
#         inputs = self.processor(
#             text=questions,
#             images=list(images),  # ensure images are passed as a list
#             return_tensors="pt",
#             padding=True,
#             image_mean=[0.5, 0.5, 0.5],
#             image_std=[0.5, 0.5, 0.5]
#         ).to(self.device)
        
#         input_ids = inputs["input_ids"]
#         pixel_values = inputs["pixel_values"]
#         labels = self.processor.tokenizer(
#             text=answers,
#             return_tensors="pt",
#             padding=True,
#             return_token_type_ids=False
#         ).input_ids.to(self.device)

#         outputs = self.model(input_ids=input_ids, pixel_values=pixel_values, labels=labels)
#         loss = outputs.loss
#         self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, batch_size=len(images), sync_dist=True)
#         torch.cuda.empty_cache()
#         return loss

#     def validation_step(self, batch, batch_idx):
#         images, questions, answers, tasks = batch
#         inputs = self.processor(
#             text=questions,
#             images=list(images),  # ensure images are passed as a list
#             return_tensors="pt",
#             padding=True,
#             
# 
# 
# 
# 
# 
# 
# 
# 
# 
# =[0.5, 0.5, 0.5],
#             image_std=[0.5, 0.5, 0.5]
#         ).to(self.device)
 
#         batch_results = evaluate_results(
#             model=self.model, 
#             inputs=inputs,
#             processor=self.processor, 
#             answers=answers, 
#             images=images,
#             batch_idx=batch_idx,
#             questions=questions
#         )

#         self.valid_outputs.append(batch_results)
#         return batch_results

#     def on_validation_epoch_end(self):
#         all_predictions = []
#         all_targets = []

#         for batch_result in self.valid_outputs:
#             all_predictions.extend(batch_result["predictions"])
#             all_targets.extend(batch_result["targets"])

#         confusion_matrix = sv.ConfusionMatrix.from_detections(
#             predictions=all_predictions, 
#             targets=all_targets, 
#             classes=CLASSES
#         )

#         mean_average_precision = sv.MeanAveragePrecision.from_detections(
#             predictions=all_predictions, 
#             targets=all_targets
#         )

#         self.log("val/mAP_50_95", mean_average_precision.map50_95)
#         self.log("val/mAP_50", mean_average_precision.map50)
#         self.log("val/mAP_75", mean_average_precision.map75)

#     def test_step(self, batch, batch_idx):
#         images, questions, answers, tasks = batch
#         inputs = self.processor(
#             text=questions,
#             images=list(images),  # Ensure images are passed as a list
#             return_tensors="pt",
#             padding=True,
#             image_mean=[0.5, 0.5, 0.5],
#             image_std=[0.5, 0.5, 0.5]
#         ).to(self.device)

#         batch_results = evaluate_results(
#             model=self.model, 
#             inputs=inputs,
#             processor=self.processor, 
#             answers=answers, 
#             images=images,
#             batch_idx=batch_idx,
#             questions=questions
#         )
#         self.test_outputs.append(batch_results)
#         return batch_results

#     def on_test_epoch_end(self):
#         all_predictions = []
#         all_targets = []

#         for batch_result in self.test_outputs:
#             all_predictions.extend(batch_result["predictions"])
#             all_targets.extend(batch_result["targets"])
        
#         confusion_matrix = sv.ConfusionMatrix.from_detections(
#             predictions=all_predictions, 
#             targets=all_targets, 
#             classes=CLASSES
#         )

#         mean_average_precision = sv.MeanAveragePrecision.from_detections(
#             predictions=all_predictions, 
#             targets=all_targets
#         )

#         print("mAP_50_95:", mean_average_precision.map50_95)
#         print("mAP_50:", mean_average_precision.map50)
#         print("mAP_75:", mean_average_precision.map75)
#         self.log("test/mAP_50_95", mean_average_precision.map50_95)
#         self.log("test/mAP_50", mean_average_precision.map50)
#         self.log("test/mAP_75", mean_average_precision.map75)

#         print("Confusion Matrix:\n", confusion_matrix.matrix)
#         print("Mean Average Precision:\n", mean_average_precision)

#     def configure_optimizers(self):
#         print('self.lr:', self.lr)
#         optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.lr)
#         lr_scheduler = get_scheduler(
#             name="linear",
#             optimizer=optimizer,
#             num_warmup_steps=0,
#             num_training_steps=self.num_training_steps,
#         )
#         return [optimizer], [lr_scheduler]


class FlorenceLightningModel(pl.LightningModule):
    def __init__(self, model, processor, lr=3e-6, num_training_steps=None):
        super().__init__()
        self.model = model
        self.processor = processor
        self.lr = float(lr)
        self.num_training_steps = num_training_steps
        self.valid_outputs = []
        self.test_outputs = []

    # plots a confusion matrix and returns it as a matplotlib figure
    def plot_confusion_matrix(matrix, class_names):
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues',
                    xticklabels=class_names,
                    yticklabels=class_names)
        ax.set_xlabel("Predicted Labels")
        ax.set_ylabel("True Labels")
        ax.set_title("Confusion Matrix")
        plt.tight_layout()
        return fig

    def training_step(self, batch, batch_idx):
        images, questions, answers, tasks, original_sizes = batch

        # print(f"Answers: {answers}")

        inputs = self.processor(
            text=questions,
            images=images,
            return_tensors="pt",
            padding=True,
            # image_mean=[0.5, 0.5, 0.5],  # no need to standardise here anymore
            # image_std=[0.5, 0.5, 0.5],
            do_rescale=False  # don't need to do normalisation in the processor, dataset already normalised
        ).to(self.device)
        
        input_ids = inputs["input_ids"]
        pixel_values = inputs["pixel_values"]

        # print(f"Input_ids: {input_ids}")

        labels = self.processor.tokenizer(
            text=answers,
            return_tensors="pt",
            padding=True,
            return_token_type_ids=False
        ).input_ids.to(self.device)

        outputs = self.model(input_ids=input_ids, pixel_values=pixel_values, labels=labels)

        loss = outputs.loss
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, batch_size=len(images), sync_dist=True)
        torch.cuda.empty_cache()
        return loss

    def validation_step(self, batch, batch_idx):
        images, questions, answers, tasks, original_sizes = batch

        # print(f"Answers: {answers}")

        inputs = self.processor(
            text=questions,
            images=images,
            return_tensors="pt",
            padding=True,
            # image_mean=[0.5, 0.5, 0.5],
            # image_std=[0.5, 0.5, 0.5],
            do_rescale=False  # don't need to do normalisation in the processor, dataset already normalised
        ).to(self.device)
 
        batch_results = evaluate_results(
            model=self.model, 
            inputs=inputs,
            processor=self.processor, 
            answers=answers, 
            images=images,
            batch_idx=batch_idx,
            questions=questions,
            original_sizes=original_sizes
        )

        print(batch_results)

        # debugging predictions and processing
        # print("Raw Model Outputs:", batch_results.get("raw_outputs", None))
        # print("Predictions:", batch_results["predictions"])
        # print("Targets:", batch_results["targets"])

        self.valid_outputs.append(batch_results)
        # print(f"Outputs at valid step: {self.valid_outputs}")
        return batch_results

    def on_validation_epoch_end(self):
        all_predictions = []
        all_targets = []

        # print(f"Outputs at valid epoch end: {self.valid_outputs}")

        for batch_result in self.valid_outputs:
            all_predictions.extend(batch_result["predictions"])
            all_targets.extend(batch_result["targets"])

        # print(f"All pred: {all_predictions}")

        # # Debug: Check prediction and target sizes
        # print("Debug: Predictions and Targets sizes:")
        # for idx, (prediction, target) in enumerate(zip(all_predictions, all_targets)):
        #     print(f"Prediction {idx}: {len(prediction.xyxy)}, {len(prediction.class_id)}")
        #     print(f"Target {idx}: {len(target.xyxy)}, {len(target.class_id)}")

        # for idx, (prediction, target) in enumerate(zip(all_predictions, all_targets)):
        #     print(f"Batch {idx}: Predictions - {prediction}, Targets - {target}")

        mean_average_precision = sv.MeanAveragePrecision.from_detections(
            predictions=all_predictions, 
            targets=all_targets
        )

        print(f"mAP details: {mean_average_precision}")

        self.log("val/mAP_50_95", mean_average_precision.map50_95)
        self.log("val/mAP_50", mean_average_precision.map50)
        self.log("val/mAP_75", mean_average_precision.map75)

        # if not all_predictions or not all_targets:
        #     print("[WARN] Empty predictions or targets after filtering.")
        #     return  # skip confusion matrix calculation if empty

        confusion_matrix = sv.ConfusionMatrix.from_detections(
            predictions=all_predictions, 
            targets=all_targets, 
            classes=CLASSES
        )

        matrix = confusion_matrix.matrix

        print("hello")
        print(all_predictions)
        print("")
        print(all_targets)

        
        # plots and logs the confusion matrix to tensorboard
        fig = self.plot_confusion_matrix(matrix, class_names=CLASSES)
        if self.logger and isinstance(self.logger, pl.loggers.TensorBoardLogger):
            self.logger.experiment.add_figure("Validation/Confusion_Matrix", fig, global_step=self.current_epoch)

        # clears the outputs, saves on memory usage
        self.valid_outputs.clear()

    def test_step(self, batch, batch_idx):
        images, questions, answers, tasks, original_sizes = batch

        inputs = self.processor(
            text=questions,
            images=images,  # ensure images are passed as a list
            return_tensors="pt",
            padding=True,
            # image_mean=[0.5, 0.5, 0.5],
            # image_std=[0.5, 0.5, 0.5],
            do_rescale=False
        ).to(self.device)
 
        batch_results = evaluate_results(
            model=self.model, 
            inputs=inputs,
            processor=self.processor, 
            answers=answers, 
            images=images,
            batch_idx=batch_idx,
            questions=questions,
            original_sizes=original_sizes
        )

        # # debugging predictions and processing
        # print("Raw Model Outputs:", batch_results.get("raw_outputs", None))
        # print("Predictions:", batch_results["predictions"])
        # print("Targets:", batch_results["targets"])

        self.valid_outputs.append(batch_results)
        return batch_results
    
    def on_test_epoch_end(self):
        all_predictions = []
        all_targets = []

        for batch_result in self.valid_outputs:
            all_predictions.extend(batch_result["predictions"])
            all_targets.extend(batch_result["targets"])

        # # Debug: Check prediction and target sizes
        # print("Debug: Predictions and Targets sizes:")
        # for idx, (prediction, target) in enumerate(zip(all_predictions, all_targets)):
        #     print(f"Prediction {idx}: {len(prediction.xyxy)}, {len(prediction.class_id)}")
        #     print(f"Target {idx}: {len(target.xyxy)}, {len(target.class_id)}")

        # # debugging predictions output
        # for idx, (prediction, target) in enumerate(zip(all_predictions, all_targets)):
        #     print(f"Batch {idx}: Predictions - {prediction}, Targets - {target}")

        mean_average_precision = sv.MeanAveragePrecision.from_detections(
            predictions=all_predictions, 
            targets=all_targets
        )

        # debugging map values
        print(f"mAP details: {mean_average_precision}")

        self.log("val/mAP_50_95", mean_average_precision.map50_95)
        self.log("val/mAP_50", mean_average_precision.map50)
        self.log("val/mAP_75", mean_average_precision.map75)

        if not all_predictions or not all_targets:
            print("[WARN] Empty predictions or targets after filtering.")
            return  # skip confusion matrix calculation if empty

        confusion_matrix = sv.ConfusionMatrix.from_detections(
            predictions=all_predictions, 
            targets=all_targets, 
            classes=CLASSES
        )

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.lr)
        scheduler = get_scheduler(
            name="linear",
            optimizer=optimizer,
            num_warmup_steps=0,
            num_training_steps=self.num_training_steps
        )
        return [optimizer], [scheduler]


In [ ]:
if config['model']['peft']['use_peft']:
    print("Using PEFT")
    # load an existing peft model checkpoint if there is one
    if config['model']['peft']['lora_checkpoint'] not in [None, "False"]:
        lora_checkpoint = LoraConfig.from_pretrained(config['model']['peft']['lora_checkpoint'])
        model = PeftModel.from_pretrained(model, lora_checkpoint, is_trainable=True)
    else:
        lora_config = LoraConfig(
                r=8,
                lora_alpha=16,
                target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "linear", "Conv2d", "lm_head", "fc2"],
                task_type="CAUSAL_LM",
                lora_dropout=0.05,
                bias="none",
                inference_mode=False,
                use_rslora=True,
                init_lora_weights="gaussian",
                revision=REVISION
            )
        model = get_peft_model(model, lora_config)
    print_trainable_parameters(model)

# else, fine tune entire language part, only freeze the vision part
elif config['model']['finetune']:
    print("Tuning language part")
    for param in model.vision_tower.parameters():
        param.requires_grad = False

# otherwise, train the entire model
else:
    print("No PEFT")
    for param in model.parameters():
        param.requires_grad = True

In [ ]:
# config['dataset']['vindr']['data_pct'] = 1.0

data_loader_manager = VinderDataLoaderManager({
    "img_root": config['dataset']['vindr']['img_root'],
    "dicom_root": config['dataset']['vindr']['dicom_root'],
    "annotation_csv": config['dataset']['vindr']['annotation_csv'],
    "batch_size": config['trainer']['train_batch_size'],
    "data_pct": config['dataset']['vindr']['data_pct'],
    "num_workers": config['trainer']['num_workers'],
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "processor": None,  # use default processor
    "use_definition": True  # change to True if want to use a definition as a prompt
})

In [ ]:
train_dataloader = data_loader_manager.train_dataloader()
val_dataloader = data_loader_manager.test_dataloader()
test_dataloader = data_loader_manager.val_dataloader()

In [ ]:
dataset_size = len(train_dataloader.dataset)
num_training_steps = (dataset_size + config['trainer']['train_batch_size'] - 1) // config['trainer']['train_batch_size']

lightning_model = FlorenceLightningModel(model=model, 
                                         processor=processor, 
                                         lr=config['trainer']['learning_rate'], 
                                         num_training_steps=num_training_steps)

In [ ]:
if config['trainer']['checkpoint_dir'] is not None:
    os.makedirs(config['trainer']['checkpoint_dir'], exist_ok=True)


custom_checkpoint_callback = CustomModelCheckpoint(
    dirpath=config['trainer']['checkpoint_dir'],
    filename='model-{epoch}-{step}',
    save_top_k=1,  # Save top 2 models based on the monitored metric
    monitor='val/mAP_50_95',  # Monitor a different metric (e.g., val_accuracy)
    mode='min',  # Mode for monitoring (min for loss, max for accuracy)
    # every_n_train_steps=500,  # every_n_train_steps >= save_top_k*val_check_interval
    save_embedding_layers=True,  # Save the embedding layers
    verbose=True  # Set to True to log when checkpoints are saved
)

In [ ]:
torch.set_float32_matmul_precision('medium')  # can be high or highest, change based on setup. Higher = better but more computationally expensive

In [ ]:
trainer = Trainer(
    max_epochs=config['trainer']['max_epochs'],
    accelerator="auto",
    # devices=1,
    devices="auto",
    strategy="auto",
    # accumulate_grad_batches=4,  # accumulate grads over 4 batches
    precision="16-mixed",  # set to mixed for faster, set to true to use less memory but more unstable model
    # log_every_n_steps=200,
    # logger=wandb_logger,
    num_sanity_val_steps=0,
    enable_checkpointing=True
    # callbacks=[custom_checkpoint_callback]
)

trainer.fit(lightning_model, train_dataloader, val_dataloader)


In [ ]:
trainer.test(lightning_model, test_dataloader)

In [ ]:
# torch.save(lightning_model.model.state_dict(), "pretrained_model.pth")

# saves the model and processor 
model.save_pretrained("pretrained_model")
processor.save_pretrained("pretrained_processor")

In [ ]:
## loads model

# lightning_model.load_state_dict(torch.load("pretrained_model.pth"))
# lightning_model.eval()

In [ ]:
# uses GPU if able
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### initialising the model
# using the pretrained model
model = AutoModelForCausalLM.from_pretrained("./pretrained_model", trust_remote_code=True).to(DEVICE)

# defines the processor to be used (florence2)
# using the pretrained processor
processor = AutoProcessor.from_pretrained("./pretrained_processor", trust_remote_code=True)

In [ ]:
# loads the cofig for running the model
config_path = "configs/experiment.yaml"
config = load_config(config_path)

# processor.image_processor.size = config['model']['processor']['image_size']
# processor.image_processor.crop_size = config['model']['processor']['crop_size']

In [ ]:
# # load pretrained weights
# state_dict = torch.load("pretrained_model.pth")
# # print(state_dict.keys())

# model.load_state_dict(state_dict, strict=False)
# model.eval()  # sets the model to evaluation mode


In [ ]:
# config['dataset']['vindr']['data_pct'] = 1.0

data_loader_manager = VinderDataLoaderManager({
    "img_root": config['dataset']['vindr']['img_root'],
    "dicom_root": config['dataset']['vindr']['dicom_root'],
    "annotation_csv": config['dataset']['vindr']['annotation_csv'],
    "batch_size": config['trainer']['train_batch_size'],
    "data_pct": config['dataset']['vindr']['data_pct'],
    "num_workers": config['trainer']['num_workers'],
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    "processor": None,  # use default processor
    "use_definition": True
})
test_dataloader = data_loader_manager.val_dataloader()

dataset_size = len(test_dataloader.dataset)
num_training_steps = (dataset_size + config['trainer']['train_batch_size'] - 1) // config['trainer']['train_batch_size']

In [ ]:


lightning_model = FlorenceLightningModel(model=model, processor=processor, lr=config['trainer']['learning_rate'], num_training_steps=num_training_steps)

if config['trainer']['checkpoint_dir'] is not None:
    os.makedirs(config['trainer']['checkpoint_dir'], exist_ok=True)

trainer = Trainer(
    max_epochs=config['trainer']['max_epochs'],
    # accelerator="xpu",
    # accelerator="cpu",
    accelerator="auto",
    # devices=1,
    devices="auto",
    strategy="auto",
    # log_every_n_steps=200,
    # logger=wandb_logger,
    num_sanity_val_steps=0,
    enable_checkpointing=True
)

trainer.test(lightning_model, test_dataloader)